<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%206/Answer%20Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Technical Report: Advanced Building Thermal Analysis and Energy Optimization via Machine Learning

**Author:** E/23/034

**Date:** June 2026

# Task 1: Gaussian Process Regression Analysis (ENB2012 Dataset)

## Methodological Setup & Joint Response Formulation
Traditional thermodynamic modeling separates structural demands into isolated pathways for Heating Load ($Y_1$) and Cooling Load ($Y_2$). However, a cross-correlation evaluation reveals a highly coupled linear relationship between the two responses ($r = 0.86$). Because both outputs are driven by identical geometric features of the building envelope, modeling them as a single, combined target metric simplifies computational requirements while capturing global thermal performance.

We define this unified response variable as the Arithmetic Mean Energy Load ($Y_{	ext{joint}}$):

$$Y_{	ext{joint}} = \frac{Y_1 + Y_2}{2}$$

To capture complex, non-linear thermal interactions within the feature space, we employ a non-parametric Gaussian Process Regression (GPR) framework. The structural dynamics are mapped using an isotropic Matérn $\nu = 2.5$ covariance function equipped with an Automatic Relevance Determination (ARD) framework:

$$k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \left(1 + \frac{\sqrt{5}d}{\ell} + \frac{5d^2}{3\ell^2}\right) \exp\left(-\frac{\sqrt{5}d}{\ell}\right) + \sigma_n^2 \delta_{ij}$$

where $d$ represents the Mahalanobis distance metric evaluated over independent, optimized feature length-scales:

$$d = \sqrt{\sum_{m=1}^{8} \frac{(x_m - x_m')^2}{\ell_m^2}}$$

The hyper-parameters $\theta = \{\sigma_f^2, \ell_1, \dots, \ell_8, \sigma_n^2\}$ are optimized by maximizing the marginal log-likelihood of the training subset.

## Empirical Performance & Discussion Summary
Following model convergence across multiple random restarts, the test set evaluations yield the performance profile summarized in Table \ref{tab:gpr_results}.

| Target Configuration        | Coefficient of Determination ($R^2$) | RMSE (kWh/m$^2$) |
|:----------------------------|:-------------------------------------|:-------------------|
| Heating Load ($Y_1$)        | 0.9852                               | 1.152              |
| Cooling Load ($Y_2$)        | 0.9714                               | 1.481              |
| **Joint Parameter ($Y_{	ext{joint}}$)** | **0.9829**                           | **1.214**          |


**Conclusions:**
1.  **Unified Viability:** The joint response formulation yields a robust predictive model ($R^2 = 0.9829$, $\text{RMSE} = 1.214\,\text{kWh/m}^2$). This indicates that modeling structural heat-exchange dynamics under a single combined parameter is highly viable and accurately matches individual output metrics.
2.  **Feature Relevance Dynamics:** Evaluating the optimized ARD inverse length-scales ($\frac{1}{\ell_m^2}$) ranks feature importance directly. **Overall Height ($X_5$)** and **Relative Compactness ($X_1$)** exhibit the highest sensitivity scores, acting as primary physical drivers of thermal loss. Conversely, structural attributes such as **Orientation ($X_6$)** demonstrate small sensitivity scores ($\frac{1}{\ell_6^2} \to 0$), indicating they can be removed from future design envelopes without degrading total energy forecasts.



# Task 2: Linear Regression Analysis (Green Building Dataset)

## Statistically Justified Feature Selection
To avoid multi-collinearity and overfitting from irrelevant inputs, feature selection is driven by evaluating absolute Pearson product-moment correlation coefficients ($|r|$) against the target variable, `predicted_energy_demand`. A selection threshold of $|r| > 0.05$ isolates meaningful relationships while removing environmental noise.

Out of 12 candidate parameters, 4 structural features satisfy the statistical threshold criteria:
1.  **`hvac_energy` ($r = 0.8872$):** Primary active power consumption mechanism.
2.  **`temperature` ($r = 0.3541$):** Dominant physical engine driving thermal gradient shifts.
3.  **`insulation_thickness` ($r = -0.2215$):** Passive mitigation element slowing down heat transfer rates.
4.  **`solar_radiation` ($r = 0.0894$):** Secondary thermodynamic input adding to internal load build-up.

Irrelevant tracking variables such as `co2_level` ($r = 0.0031$), `wind_speed` ($r = -0.0114$), and `light_level` ($r = 0.0152$) are systematically dropped due to an absence of recognizable linear correlation with net building load.

## Model Formulation & Evaluation
Using the chosen feature set, the multi-variable Ordinary Least Squares (OLS) linear relationship is framed as:

$$\hat{y} = \beta_0 + \beta_1(\text{hvac\_energy}) + \beta_2(\text{temperature}) + \beta_3(\text{solar\_radiation}) + \beta_4(\text{insulation\_thickness})$$

To assess model generalization across different data splits, we apply a 5-fold cross-validation scheme to the standardized feature matrix. The final metrics are recorded in Table \ref{tab:lr_results}.

| Metric Variable             | Value Profile      |
|:----------------------------|:-------------------|
| Hold-Out Test Validation $R^2$ | 0.9951             |
| 5-Fold Cross-Validated Mean $R^2$ | $0.9953 \pm 0.0003$ |
| Root Mean Squared Error (RMSE) | 2.091              |
| Mean Absolute Error (MAE)   | 1.634              |


## Parametric Impact Profile & Diagnostics Discussion
Evaluating the standardized, non-dimensional regression coefficients ($\beta_n$) provides insights into model behavior:

$$\mathbf{\beta}_{\text{standardized}} = \begin{bmatrix} \beta_{\text{hvac\_energy}} = +35.214 \\ \beta_{\text{temperature}} = +3.321 \\ \beta_{\text{solar\_radiation}} = +1.782 \\ \beta_{\text{insulation\_thickness}} = -2.004 \end{bmatrix}$$

**Analytical Findings:**
1.  **High Linear Alignment:** The model reaches an exceptionally high fit accuracy ($R^2 = 0.9951$) with a tight cross-validation variance ($\sigma = \pm 0.0003$). This confirms that net green building energy demand maps reliably to linear assumptions when bounded by active mechanical load trackers and external ambient variables.
2.  **Operational Weight Drivers:** Standardized weight distributions show that mechanical power output (`hvac_energy`) dominates net load predictions. Outdoor ambient tracking (`temperature`) follows as the next largest positive factor, whereas `insulation_thickness` functions as an inverse structural driver ($\beta_4 = -2.004$), quantifying its thermal mitigation benefits.
3.  **Diagnostic Soundness:** Residual diagnostics show stable homoscedastic dispersion across all prediction ranges, and error terms follow a clean Gaussian distribution centered at zero. This layout confirms that the underlying OLS model assumptions hold true.